In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load cleaned fraud data (from task-1-data-cleaning)
df = pd.read_csv('../data/processed/fraud_data_cleaned.csv')
print("Loaded shape:", df.shape)

Loaded shape: (151112, 11)


In [17]:
# Load IP mapping
ip_map = pd.read_csv('../data/processed/ip_country_cleaned.csv')
ip_map['lower'] = pd.to_numeric(ip_map['lower_bound_ip_address'], errors='coerce')
ip_map['upper'] = pd.to_numeric(ip_map['upper_bound_ip_address'], errors='coerce')
ip_map = ip_map.dropna(subset=['lower', 'upper'])
intervals = pd.IntervalIndex.from_arrays(ip_map['lower'], ip_map['upper'], closed='both')

def ip_to_int_safe(ip):
    if not isinstance(ip, str):
        return None
    parts = ip.split('.')
    if len(parts) != 4:
        return None
    try:
        return int(parts[0]) * 256**3 + int(parts[1]) * 256**2 + int(parts[2]) * 256 + int(parts[3])
    except:
        return None

def get_country_safe(ip_int):
    if ip_int is None:
        return 'Unknown'
    idx = intervals.get_indexer([ip_int])[0]
    if idx == -1:
        return 'Unknown'
    return ip_map.iloc[idx]['country']

df['ip_int'] = df['ip_address'].astype(str).apply(ip_to_int_safe)
df['country'] = df['ip_int'].apply(get_country_safe)
# Fill any None ip_int with 0 (so that StandardScaler works)
df['ip_int'] = df['ip_int'].fillna(0).astype(int)

print("Geolocation complete. Rows retained:", len(df))
print(df[['ip_address', 'ip_int', 'country']].head())

Geolocation complete. Rows retained: 151112
     ip_address  ip_int  country
0  7.327584e+08       0  Unknown
1  3.503114e+08       0  Unknown
2  2.621474e+09       0  Unknown
3  3.840542e+09       0  Unknown
4  4.155831e+08       0  Unknown


In [18]:
# Convert to datetime
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

# Temporal features
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 3600

# Transaction velocity (count per user)
velocity = df.groupby('user_id')['purchase_time'].count().rename('velocity')
df = df.merge(velocity, left_on='user_id', right_index=True)

print("Temporal features and velocity added. Shape:", df.shape)

Temporal features and velocity added. Shape: (151112, 17)


In [19]:
features = ['purchase_value', 'age', 'time_since_signup', 'hour_of_day', 'velocity', 'ip_int']
categorical = ['source', 'browser', 'sex', 'country']

# Ensure all columns exist (they should)
missing = [c for c in features + categorical if c not in df.columns]
if missing:
    print("Missing columns:", missing)
else:
    print("All columns present.")

X = df[features + categorical]
y = df['class']
print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

All columns present.
X shape: (151112, 10)
y distribution:
 class
0    136961
1     14151
Name: count, dtype: int64


In [20]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])

X_processed = preprocessor.fit_transform(X)
print("Processed X shape:", X_processed.shape)

Processed X shape: (151112, 17)


In [21]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Before SMOTE:\n", y_train.value_counts())

Train size: (120889, 17)
Test size: (30223, 17)
Before SMOTE:
 class
0    109568
1     11321
Name: count, dtype: int64


In [22]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("After SMOTE:\n", pd.Series(y_train_resampled).value_counts())

After SMOTE:
 class
0    109568
1    109568
Name: count, dtype: int64


In [23]:
np.save('../data/processed/X_train.npy', X_train_resampled)
np.save('../data/processed/y_train.npy', y_train_resampled)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_test.npy', y_test)
print("Saved preprocessed data to data/processed/")

Saved preprocessed data to data/processed/
